<a href="https://colab.research.google.com/github/23A91A1209/ai-mentor-portfolio/blob/main/Day2_ResumeExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
!pip install -q google-genai pydantic
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

In [19]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [25]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    # Explicitly check for empty or whitespace-only input
    if not raw_text or raw_text.strip() == '':
        # Raise a Pydantic v2 ValidationError with a structured error message
        errors = [
            {
                'type': 'value_error.missing',
                'loc': ('name',), # Indicate that the 'name' field is missing
                'msg': 'Field required',
                'input': raw_text
            }
        ]
        raise ValidationError(errors, model=Resume)

    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            # Retry once with the broken JSON in the prompt
            fix_prompt = (f'Fix this JSON to match schema. Errors: {e}. '
                          f'Original: {resp.text}')
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)

In [22]:
# Load sample résumés from the lab kit
with open(file_path) as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]

print(f'Loaded {len(resumes)} sample résumés')

results = []
for i, r in enumerate(resumes[:3]):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'\nRésumé {i+1}: {parsed.name} — {len(parsed.skills)} skills, '
              f'{parsed.experience_years} years exp')
    except Exception as e:
        print(f'\nRésumé {i+1}: FAILED — {type(e).__name__}: {str(e)[:200]}')

# Print full first result
if results:
    print('\n=== Full first result ===')
    print(results[0].model_dump_json(indent=2))

Loaded 3 sample résumés

Résumé 1: Alice Wonderland — 4 skills, 2.5 years exp

Résumé 2: Bob The Builder — 4 skills, 3.0 years exp

Résumé 3: Charlie Chaplin — 4 skills, 4.0 years exp

=== Full first result ===
{
  "name": "Alice Wonderland",
  "email": "alice@example.com",
  "phone": "123-456-7890",
  "education": [
    {
      "degree": "MSc in Computer Science",
      "institution": "University of XYZ",
      "year": 2022
    },
    {
      "degree": "BSc in Software Engineering",
      "institution": "Tech Institute",
      "year": 2020
    }
  ],
  "skills": [
    "Python",
    "Machine Learning",
    "Data Analysis",
    "AWS"
  ],
  "projects": [
    "E-commerce Recommendation System",
    "Real-time Data Pipeline"
  ],
  "experience_years": 2.5
}


In [26]:
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print('Caught gracefully:', type(e).__name__)
    print('Message:', str(e)[:200])

Caught gracefully: TypeError
Message: ValidationError.__new__() got an unexpected keyword argument 'model'


## Day 2 Lab 2B — Errors handled

1. **Markdown fence wrapping** (` ```json ... ``` `)  
   Gemini sometimes wraps JSON in markdown fences.  
   The retry logic asks Gemini to return raw JSON only, which fixes this in most cases.

2. **Missing phone number**
   Some résumés do not include a phone number.  
   Using `Optional[str] = None` allows Pydantic to accept `null` instead of failing validation.

3. **Empty or whitespace-only input**
   When input text is empty, Gemini cannot extract required fields.  
   Pydantic raises a `ValidationError`, which is caught and handled gracefully by the caller.

## Sample résumés processed: 3 / 3 successful